<a href="https://colab.research.google.com/github/muajnstu/CAST/blob/main/Perception.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
import os
import io
import math
import csv
import pandas as pd
import numpy as np
import warnings
from google.colab import files
warnings.filterwarnings('ignore')

# Core ML
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder
from sklearn.feature_selection import SelectKBest, f_classif, mutual_info_classif, RFE
from sklearn.metrics import (
    accuracy_score, classification_report,
    matthews_corrcoef, roc_auc_score
)
from imblearn.metrics import geometric_mean_score
from sklearn.model_selection import StratifiedKFold

# Classifiers
from sklearn.ensemble import (
    RandomForestClassifier, ExtraTreesClassifier, BaggingClassifier,
    GradientBoostingClassifier, VotingClassifier, StackingClassifier
)
from sklearn.linear_model import (
    LogisticRegression, RidgeClassifier, Perceptron,
    SGDClassifier, PassiveAggressiveClassifier
)
from sklearn.tree import DecisionTreeClassifier
from sklearn.naive_bayes import GaussianNB
from sklearn.discriminant_analysis import (
    LinearDiscriminantAnalysis, QuadraticDiscriminantAnalysis
)
from lightgbm import LGBMClassifier
from xgboost import XGBClassifier

# Balancing
from imblearn.over_sampling import RandomOverSampler, SMOTE, ADASYN, BorderlineSMOTE
from imblearn.under_sampling import RandomUnderSampler, TomekLinks
from imblearn.combine import SMOTETomek


# optimization
from sklearn.model_selection import cross_val_score, StratifiedKFold
from sklearn.feature_selection import RFECV


In [2]:

df = pd.read_csv(
    'https://raw.githubusercontent.com/muajnstu/CAST/refs/heads/main/Perception/Perception_categorized.csv'
)
print(f"Raw dataset shape : {df.shape}")
print(f"Target distribution:\n{df['CAST'].value_counts()}\n")

X_raw = df.drop(columns=['CAST'])
y_raw = df['CAST']

Raw dataset shape : (400, 52)
Target distribution:
CAST
1    161
2    152
0     87
Name: count, dtype: int64



In [3]:
def evaluate_dataset(data, target_col='CAST', label="", verbose=True, n_splits=10):
    X_ = data.drop(columns=[target_col])
    y_ = data[target_col]

    skf = StratifiedKFold(n_splits=n_splits, shuffle=True, random_state=42)

    rows = []
    for clf_name, model in build_models().items():
        fold_metrics = {
            'Accuracy': [], 'Macro_F1': [], 'Macro_Precision': [],
            'Macro_Recall': [], 'MCC': [], 'G_Mean': [], 'AUC': []
        }

        for train_idx, test_idx in skf.split(X_, y_):
            X_tr, X_te = X_.iloc[train_idx], X_.iloc[test_idx]
            y_tr, y_te = y_.iloc[train_idx], y_.iloc[test_idx]

            try:
                model.fit(X_tr, y_tr)
                y_pred = model.predict(X_te)

                acc   = accuracy_score(y_te, y_pred)
                rep   = classification_report(y_te, y_pred, zero_division=0, output_dict=True)
                mcc   = matthews_corrcoef(y_te, y_pred)
                gmean = geometric_mean_score(y_te, y_pred, average='macro')

                n_classes = len(np.unique(y_te))
                auc = None
                try:
                    if hasattr(model, 'predict_proba'):
                        y_score = model.predict_proba(X_te)
                    else:
                        y_score = model.decision_function(X_te)
                        if n_classes <= 2:
                            y_score = y_score.reshape(-1, 1)

                    if n_classes == 2:
                        prob = y_score[:, 1] if y_score.ndim == 2 else y_score
                        auc = roc_auc_score(y_te, prob)
                    else:
                        auc = roc_auc_score(
                            y_te, y_score,
                            multi_class='ovr', average='macro',
                            labels=np.unique(y_te)
                        )
                except Exception:
                    pass

                fold_metrics['Accuracy'].append(acc)
                fold_metrics['Macro_F1'].append(rep['macro avg']['f1-score'])
                fold_metrics['Macro_Precision'].append(rep['macro avg']['precision'])
                fold_metrics['Macro_Recall'].append(rep['macro avg']['recall'])
                fold_metrics['MCC'].append(mcc)
                fold_metrics['G_Mean'].append(gmean)
                if auc is not None:
                    fold_metrics['AUC'].append(auc)

            except Exception as e:
                if verbose:
                    print(f"    {clf_name:25s}  Fold ERROR: {e}")

        # Aggregate: mean ± std across folds
        def fmt(vals):
            if not vals:
                return None, None
            return round(np.mean(vals), 3), round(np.std(vals), 3)

        acc_m,   acc_s   = fmt(fold_metrics['Accuracy'])
        f1_m,    f1_s    = fmt(fold_metrics['Macro_F1'])
        prec_m,  prec_s  = fmt(fold_metrics['Macro_Precision'])
        rec_m,   rec_s   = fmt(fold_metrics['Macro_Recall'])
        mcc_m,   mcc_s   = fmt(fold_metrics['MCC'])
        gm_m,    gm_s    = fmt(fold_metrics['G_Mean'])
        auc_m,   auc_s   = fmt(fold_metrics['AUC'])

        rows.append({
            'Label':                label,
            'Classifier':           clf_name,
            'Accuracy':             acc_m,  'Accuracy_SD':   acc_s,
            'Macro_F1':             f1_m,   'Macro_F1_SD':   f1_s,
            'Macro_Precision':      prec_m, 'Macro_Prec_SD': prec_s,
            'Macro_Recall':         rec_m,  'Macro_Rec_SD':  rec_s,
            'MCC':                  mcc_m,  'MCC_SD':        mcc_s,
            'G_Mean':               gm_m,   'G_Mean_SD':     gm_s,
            'AUC':                  auc_m,  'AUC_SD':        auc_s,
        })

        if verbose:
            auc_str = f"{auc_m:.3f}±{auc_s:.3f}" if auc_m is not None else "N/A"
            print(f"    {clf_name:25s}  "
                  f"Acc={acc_m:.3f}±{acc_s:.3f}  "
                  f"F1={f1_m:.3f}±{f1_s:.3f}  "
                  f"MCC={mcc_m:.3f}±{mcc_s:.3f}  "
                  f"G-Mean={gm_m:.3f}±{gm_s:.3f}  "
                  f"AUC={auc_str}")

    return rows

In [4]:
def build_models():
    return {
        "RandomForest":       RandomForestClassifier(random_state=42),
        "ExtraTrees":         ExtraTreesClassifier(random_state=42),
        "Bagging":            BaggingClassifier(random_state=42),
        "GradientBoosting":   GradientBoostingClassifier(random_state=42),
        "LogisticRegression": LogisticRegression(max_iter=1000, random_state=42),
        "RidgeClassifier":    RidgeClassifier(),
        "DecisionTree":       DecisionTreeClassifier(random_state=42),
        "NaiveBayes":         GaussianNB(),
        "Perceptron":         Perceptron(random_state=42),
        "SGDClassifier":      SGDClassifier(random_state=42),
        "PassiveAggressive":  PassiveAggressiveClassifier(random_state=42),
        "LDA":                LinearDiscriminantAnalysis(),
        "QDA":                QuadraticDiscriminantAnalysis(),
        "LightGBM":           LGBMClassifier(verbosity=-1, random_state=42),
        "XGBoost":            XGBClassifier(n_estimators=100, random_state=42,
                                            eval_metric='mlogloss',
                                            use_label_encoder=False, verbosity=0),
        "VotingSoft": VotingClassifier(estimators=[
            ('rf',   RandomForestClassifier(n_estimators=100, random_state=42)),
            ('gb',   GradientBoostingClassifier(n_estimators=100, random_state=42)),
            ('et',   ExtraTreesClassifier(n_estimators=100, random_state=42)),
            ('lgbm', LGBMClassifier(verbosity=-1, n_estimators=100, random_state=42)),
        ], voting='soft', n_jobs=-1),
        "VotingHard": VotingClassifier(estimators=[
            ('rf',   RandomForestClassifier(n_estimators=100, random_state=42)),
            ('et',   ExtraTreesClassifier(n_estimators=100, random_state=42)),
            ('lgbm', LGBMClassifier(verbosity=-1, n_estimators=100, random_state=42)),
        ], voting='hard', n_jobs=-1),
        "Stacking": StackingClassifier(estimators=[
            ('rf',   RandomForestClassifier(n_estimators=100, random_state=42)),
            ('gb',   GradientBoostingClassifier(n_estimators=100, random_state=42)),
            ('et',   ExtraTreesClassifier(n_estimators=100, random_state=42)),
            ('lgbm', LGBMClassifier(verbosity=-1, n_estimators=100, random_state=42)),
        ], final_estimator=LogisticRegression(max_iter=1000, random_state=42),
           cv=5, stack_method='predict_proba', n_jobs=-1)
    }

In [6]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    X_raw, y_raw,
    test_size=0.2,
    random_state=42,
    stratify=y_raw
)

In [7]:
def fmt_metric(mean_val, std_val):
    """Format mean±std, returning N/A if None or NaN."""
    try:
        if mean_val is None or std_val is None:
            return "N/A"
        if math.isnan(float(mean_val)) or math.isnan(float(std_val)):
            return "N/A"
        return f"{float(mean_val):.3f}\u00b1{float(std_val):.3f}"  # \u00b1 = ±
    except (TypeError, ValueError):
        return "N/A"

COLUMNS = ["Model",
           f"Accuracy (Mean \u00b1 SD)",
           f"MCC (Mean \u00b1 SD)",
           f"G-Mean (Mean \u00b1 SD)",
           f"F1-Score (Mean \u00b1 SD)",
           f"AUROC (Mean \u00b1 SD)"]

output = io.StringIO()
writer = csv.writer(output)

In [8]:
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score
import numpy as np

acc_list, prec_list, rec_list, f1_list = [], [], [], []

In [9]:
results = evaluate_dataset(df)
result_df = pd.DataFrame(results)
result_df = result_df.sort_values(by="Macro_F1", ascending=False)

result_df

    RandomForest               Acc=0.928±0.032  F1=0.928±0.034  MCC=0.889±0.050  G-Mean=0.942±0.028  AUC=0.990±0.007
    ExtraTrees                 Acc=0.915±0.042  F1=0.915±0.043  MCC=0.869±0.066  G-Mean=0.931±0.034  AUC=0.989±0.009
    Bagging                    Acc=0.935±0.025  F1=0.938±0.024  MCC=0.902±0.039  G-Mean=0.951±0.017  AUC=0.989±0.009
    GradientBoosting           Acc=0.932±0.035  F1=0.931±0.037  MCC=0.898±0.054  G-Mean=0.949±0.025  AUC=0.993±0.006
    LogisticRegression         Acc=0.892±0.034  F1=0.896±0.033  MCC=0.835±0.052  G-Mean=0.916±0.031  AUC=0.980±0.011
    RidgeClassifier            Acc=0.738±0.092  F1=0.732±0.093  MCC=0.607±0.146  G-Mean=0.801±0.077  AUC=N/A
    DecisionTree               Acc=0.898±0.024  F1=0.896±0.024  MCC=0.844±0.037  G-Mean=0.922±0.020  AUC=0.923±0.020
    NaiveBayes                 Acc=0.810±0.069  F1=0.807±0.071  MCC=0.710±0.107  G-Mean=0.852±0.054  AUC=0.939±0.029
    Perceptron                 Acc=0.675±0.050  F1=0.665±0.071  MCC=0.53

,Label,Classifier,Accuracy,Accuracy_SD,Macro_F1,Macro_F1_SD,Macro_Precision,Macro_Prec_SD,Macro_Recall,Macro_Rec_SD,MCC,MCC_SD,G_Mean,G_Mean_SD,AUC,AUC_SD
13,,LightGBM,0.940,0.028,0.939,0.029,0.945,0.028,0.937,0.030,0.908,0.043,0.953,0.022,0.993,0.006
2,,Bagging,0.935,0.025,0.938,0.024,0.943,0.027,0.938,0.021,0.902,0.039,0.951,0.017,0.989,0.009
15,,VotingSoft,0.940,0.028,0.938,0.029,0.945,0.028,0.936,0.030,0.908,0.043,0.952,0.022,0.992,0.006
17,,Stacking,0.938,0.028,0.936,0.029,0.943,0.029,0.934,0.030,0.905,0.043,0.950,0.022,0.992,0.006
14,,XGBoost,0.935,0.028,0.934,0.035,0.943,0.029,0.933,0.037,0.902,0.042,0.949,0.026,0.992,0.007
3,,GradientBoosting,0.932,0.035,0.931,0.037,0.934,0.040,0.933,0.034,0.898,0.054,0.949,0.025,0.993,0.006
0,,RandomForest,0.928,0.032,0.928,0.034,0.939,0.027,0.923,0.038,0.889,0.050,0.942,0.028,0.990,0.007
16,,VotingHard,0.928,0.036,0.927,0.040,0.940,0.029,0.921,0.045,0.889,0.056,0.940,0.032,NaN,NaN
1,,ExtraTrees,0.915,0.042,0.915,0.043,0.929,0.037,0.908,0.046,0.869,0.066,0.931,0.034,0.989,0.009
11,,LDA,0.892,0.045,0.898,0.046,0.915,0.035,0.892,0.051,0.836,0.070,0.916,0.038,0.983,0.011
